# 09. Structured & Record Arrays: Beginner Guide

### 🌟 What Are Structured & Record Arrays in NumPy?
While standard NumPy arrays are homogeneous (one data type), **Structured Arrays** allow you to define compound C-struct data types with named fields (e.g. an integer ID, float amount, and string code) packed tightly in binary memory.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Dtype Construction (`np.dtype`)**: Defining multi-type fields using `np.dtype([('name', 'U10'), ('age', 'i4'), ('weight', 'f4')])`.
- **Field Access**: Referencing columns by field name (`structured_arr['age']`).
- **Record Arrays (`np.recarray`)**: Enabling dot-notation attribute access (`arr.age`).
- **Memory Alignment (`align=True`)**: Compiler padding bytes for C-struct hardware alignment.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from ../data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### 🔹 Dtype Construction with `np.dtype`
Constructs a C-struct array holding transaction ID, amount, and fraud flag in a single contiguous binary buffer. Checking and validating data types prevents subtle runtime errors and ensures subsequent mathematical or string operations behave properly. **Tip:** Prefer `isinstance(obj, ClassName)` over `type(obj) == ClassName` when checking types in conditional logic.

**Syntax:** `np.dtype([('tx_id', 'U10'), ('amount', 'f8'), ('is_fraud', 'i1')])`


In [2]:
tx_struct_dtype = np.dtype([
    ('tx_id', 'U12'),
    ('amount', np.float64),
    ('is_fraud', np.int8)
])
struct_records = np.array([
    (clean_raw['transaction_id'][i], clean_raw['transaction_amount'][i], clean_raw['is_fraud'][i])
    for i in range(5)
], dtype=tx_struct_dtype)
print('Structured Transactions Array:\n', struct_records)

Structured Transactions Array:
 [('TX110686', 1216.33, 0) ('TX107170',  324.99, 0)
 ('TX108328',  136.66, 0) ('TX108563',  124.21, 0)
 ('TX107002', 1284.68, 0)]


### 🔹 Field Access: `arr['field']`
Extracts `amount` and `is_fraud` fields as zero-copy views. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `struct_records['amount']`


In [3]:
print('Amount Field View:', struct_records['amount'])
print('Fraud Field View:', struct_records['is_fraud'])

Amount Field View: [1216.33  324.99  136.66  124.21 1284.68]
Fraud Field View: [0 0 0 0 0]


### 🔹 Record Arrays with Dot Notation (`np.recarray`)
Enables dot-notation attribute access (`rec.amount`) over structured memory. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `rec = struct_records.view(np.recarray)`


In [4]:
rec_tx = struct_records.view(np.recarray)
print('Dot Notation Access (rec_tx.tx_id):', rec_tx.tx_id)
print('Dot Notation Access (rec_tx.amount):', rec_tx.amount)

Dot Notation Access (rec_tx.tx_id): ['TX110686' 'TX107170' 'TX108328' 'TX108563' 'TX107002']
Dot Notation Access (rec_tx.amount): [1216.33  324.99  136.66  124.21 1284.68]


### 🔹 Memory Alignment with `align=True`
Pads composite C-structs for 64-bit hardware alignment. NumPy runs in compiled C memory buffers, enabling mathematical operations across millions of numbers simultaneously in milliseconds. **Tip:** NumPy operations are optimized for homogeneous numeric data, offering massive speed improvements over standard Python loops.

**Syntax:** `np.dtype([('flag', 'i1'), ('amt', 'f8')], align=True)`


In [5]:
unaligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')])
aligned_dt = np.dtype([('flag', 'i1'), ('amt', 'f8')], align=True)
print('Unaligned itemsize:', unaligned_dt.itemsize, 'bytes')
print('Aligned itemsize (padded):', aligned_dt.itemsize, 'bytes')

Unaligned itemsize: 9 bytes
Aligned itemsize (padded): 16 bytes


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Filtering Structured Transaction Records

**Approach:** Filter structured transaction records where `amount > 200` and `is_fraud == 0`.
**Syntax:** `struct_records[(struct_records['amount'] > 200) & (struct_records['is_fraud'] == 0)]`


In [6]:
filtered_records = struct_records[(struct_records['amount'] > 100) & (struct_records['is_fraud'] == 0)]
print('Filtered Valid Records:\n', filtered_records)

Filtered Valid Records:
 [('TX110686', 1216.33, 0) ('TX107170',  324.99, 0)
 ('TX108328',  136.66, 0) ('TX108563',  124.21, 0)
 ('TX107002', 1284.68, 0)]
